# Lab 05 — Support Vector Machines and Neural Networks

**Class 05 · Topic 4.** Guide: `practice/05_svm_neural_networks.pdf`. Lecture: `slides/05_svm_neural_networks.pdf`. Demos:
`01_svm.ipynb` and `02_neural_networks.ipynb`.

Cells marked **TODO** contain `raise NotImplementedError`: replace it with your code. Every other cell is ready: it draws
the plots and prints the comparisons once your functions work.

**The rule of this lab.** The SVM, the kernel SVM, the SVR and the MLP of parts B-E are *losses written with `jax.numpy`*;
the gradient always comes from `jax.grad` (inside the given `train`). Every model is then compared with its scikit-learn
or Keras twin on the **same split and the same scaling**.

In [ ]:
import time
import warnings

import jax
import jax.numpy as jnp
import keras
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from lab05_utils import (
    compare,
    fit_standardizer,
    load_breast_cancer,
    load_digits,
    plot_decision,
    plot_digits,
    plot_history,
    plot_scores_grid,
    standardize,
    stratified_split,
    train,
)
from scipy import stats
from sklearn.datasets import make_blobs, make_moons
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics.pairwise import polynomial_kernel as sk_poly_kernel
from sklearn.metrics.pairwise import rbf_kernel as sk_rbf_kernel
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC, SVR

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (9, 4)
pl.Config.set_tbl_rows(30)
pl.Config.set_float_precision(3)
pl.Config.set_tbl_formatting("ASCII_MARKDOWN")
_ = pl.Config.set_tbl_hide_dataframe_shape(True)

# Part A — Data visualization and preparation

Three data sets: **breast cancer** (30 measurements of cell nuclei, 569 tumours, two classes), the handwritten **digits**
(8x8 images, 1797, ten classes) and two **moons** (a 2D problem that a line cannot solve). The split is stratified and the
standardization statistics come from the **training** part only (Class 03).

In [ ]:
Xb, yb, names_b = load_breast_cancer()
Xd, yd, _ = load_digits()
print(f"breast cancer: {Xb.shape}, benign {yb.mean():.1%}; digits: {Xd.shape}, classes {np.bincount(yd)}")
rng = np.random.default_rng(0)
tr, te = stratified_split(yb, 0.25, rng)
mu, sd = fit_standardizer(Xb[tr])
Xb_tr, Xb_te = standardize(Xb[tr], mu, sd), standardize(Xb[te], mu, sd)
yb_tr, yb_te = yb[tr], yb[te]
spread = pl.DataFrame({"feature": names_b[:5], "min": Xb[:, :5].min(0), "max": Xb[:, :5].max(0), "std": Xb[:, :5].std(0)})
print(spread)

Xm, ym = make_moons(200, noise=0.25, random_state=0)
Xm_te, ym_te = make_moons(2000, noise=0.25, random_state=1)
tr_d, te_d = stratified_split(yd, 0.25, np.random.default_rng(0))
Xd_tr, Xd_te, yd_tr, yd_te = Xd[tr_d], Xd[te_d], yd[tr_d], yd[te_d]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for k in (0, 1):
    axes[0].scatter(Xb_tr[yb_tr == k, 0], Xb_tr[yb_tr == k, 1], s=10, label=("malignant", "benign")[k])
axes[0].set(xlabel=f"{names_b[0]} (standardized)", ylabel=f"{names_b[1]} (standardized)", title="breast cancer, two of 30 features")
axes[0].legend()
axes[1].scatter(*Xm.T, c=ym, cmap="coolwarm", s=12)
axes[1].set_title("moons")
corr = np.corrcoef(Xb_tr.T)
im = axes[2].imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
axes[2].set_title("breast cancer: correlation of the 30 features")
plt.colorbar(im, ax=axes[2])
plt.tight_layout()
plt.show()
plot_digits(Xd, yd)

**Answers (A).**

1. The benign class is 62.7%: always predicting "benign" gives accuracy **0.627**. A model at 0.979 removes about 94% of that baseline's errors (37.3% -> 2.1%). Accuracy alone hides the cost: a missed malignant tumour is the expensive error (Class 03 metrics).
2. `mean_area` has a standard deviation of 351.6 and `mean_smoothness` of 0.014: a ratio of about 25 000, so about 6e8 in squared distance. Without scaling the RBF kernel and the penalty $\lVert w\rVert^2$ see only the area (and the perimeter): the other 28 features are ignored.
3. Radius, perimeter and area are almost perfectly correlated (the block of dark red squares): the same information three times. A linear model can split a weight among them in many ways, so single weights are unstable and hard to interpret; the SVM penalty shares the weight among them.

# Part B — The linear SVM as a loss

With labels $y_i \in \{-1, +1\}$ and $f(x) = w^\top x + b$, the soft-margin SVM minimizes the **hinge loss plus an $L_2$
penalty**:

$$\mathcal{L}(w, b) = \frac1n\sum_i \max\big(0,\ 1 - y_i f(x_i)\big) + \frac{\lambda}{2}\lVert w\rVert^2,
\qquad \lambda = \frac{1}{C\,n}.$$

## B1. The loss and its support vectors

Implement `hinge_loss(params, X, y, lam)` with `params = {"w": ..., "b": ...}` (no gradient formula: `jax.grad` is called
by `train`), `margins(params, X, y)` $= y_i f(x_i)$, and `support_mask(params, X, y, tol)`: the points with margin $\le 1 +$ `tol`.

In [ ]:
def hinge_loss(params, X, y, lam):
    return jnp.mean(jnp.maximum(0.0, 1.0 - margins(params, X, y))) + 0.5 * lam * jnp.sum(params["w"] ** 2)


def margins(params, X, y):
    return y * (X @ params["w"] + params["b"])


def support_mask(params, X, y, tol=1e-3):
    return np.asarray(margins(params, X, y) <= 1.0 + tol)

**How we got there.** The slack of a point is $\xi_i = \max(0, 1 - m_i)$ with the margin $m_i = y_i f(x_i)$: zero when the
point is correct with room to spare, growing linearly when it enters the margin or crosses the line. The mean of the
slacks plus the penalty is the loss; `jnp.maximum` has a subgradient (0 where $m > 1$, $-y\,x$ where $m < 1$) and that is
what `jax.grad` returns. A support vector is a point whose constraint is active ($m \le 1$): slack $>0$ or exactly on the margin.

The driver trains the SVM for three values of $C$ (`lam = 1 / (C n)`) on the breast-cancer data and compares the
weights, the predictions and the number of support vectors with `SVC(kernel="linear")`, a quadratic-programming solver.

In [ ]:
ypm_tr = 2.0 * yb_tr - 1
ypm_te = 2.0 * yb_te - 1
rows = []
for C in (0.01, 0.1, 1, 10):
    lam = 1 / (C * len(yb_tr))
    p0 = {"w": jnp.zeros(Xb_tr.shape[1]), "b": jnp.zeros(())}
    p, hist = train(hinge_loss, p0, Xb_tr, ypm_tr, lam, lr=0.05, steps=3000)
    ref = SVC(kernel="linear", C=C, tol=1e-8).fit(Xb_tr, yb_tr)
    f_te = np.asarray(Xb_te @ p["w"] + p["b"])
    rows.append((C, float(np.abs(np.asarray(p["w"]) - ref.coef_[0]).max()), float(np.mean((f_te > 0) == ref.predict(Xb_te))),
                 int(support_mask(p, Xb_tr, ypm_tr).sum()), len(ref.support_), float(np.mean((f_te > 0) == yb_te))))
print(pl.DataFrame(rows, schema=["C", "max |w - w_svc|", "same predictions", "SV (ours)", "SV (SVC)", "test accuracy"], orient="row"))

**Answers (B1-B3).**

1. Margins 2, 0.5, -1 give hinge losses 0, 0.5 and 2. The subgradient with respect to $w$ is $0$ for the first (margin $> 1$) and $-y\,x$ for the other two; the hinge is flat for well-classified points, unlike the squared error.
2. The number of support vectors falls as $C$ grows: 102, 54, 34, 28 for $C = 0.01, 0.1, 1, 10$ (SVC: 102, 55, 35, 32). A large $C$ makes every violation expensive, the margin narrows and fewer points sit inside it. Our count and SVC's differ by 0-4 points: a point at margin 1.0005 is counted by SVC and not by our tolerance (`tol = 1e-3`) and the opposite happens for a point that gradient descent has not pushed fully out.
3. For $C \le 1$ the weights agree to 1e-3 and the predictions on all test points. For $C = 10$ the weights differ by 0.70 and 0.7% of the predictions: the loss becomes almost flat along some directions (a nearly separable data set, small $\lambda$), the Adam run with a fixed number of steps stops in a different point of the flat valley, and the solver stops with its own tolerance. The *objective values* are close; the minimizer is not unique in practice. The test accuracy is 0.979 for the first three values of $C$ and 0.965 for $C = 10$ (see the next cell: the best $C$ is 0.01-1).

**The picture in two dimensions.** Three values of $C$ on two Gaussian blobs: the dashed lines are $f = \pm 1$; the rings
are the support vectors of your loss.

In [ ]:
X2, y2 = make_blobs(120, centers=[(-1.0, -0.8), (1.0, 0.8)], cluster_std=1.2, random_state=1)
y2pm = 2.0 * y2 - 1
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, C in zip(axes, (0.01, 1, 100)):
    p, _ = train(hinge_loss, {"w": jnp.zeros(2), "b": jnp.zeros(())}, X2, y2pm, 1 / (C * len(y2)), lr=0.05, steps=3000)
    sv = support_mask(p, X2, y2pm)
    w_, b_ = np.asarray(p["w"]), float(p["b"])
    plot_decision(lambda P, w_=w_, b_=b_: P @ w_ + b_, X2, y2, f"C = {C:g}: {sv.sum()} support vectors, width {2 / np.linalg.norm(w_):.2f}",
                  ax=ax, sv=X2[sv], margin=True)
plt.tight_layout()
plt.show()

**Answers (B4).** The titles give the width and the number of support vectors. A small $C$ (0.01) gives a wide margin with most of the 120 points inside or on it; $C = 100$ gives a narrow margin defined by the few points near the overlap, and the line follows the overlapping region more closely. The line itself moves little: $C$ mainly changes how many points matter.

# Part C — Kernels

A kernel $k(a, b)$ is a similarity. With the Gram matrix $K_{ij} = k(x_i, x_j)$, the SVM in its **representer form** is

$$f(x) = \sum_j \beta_j\, k(x_j, x) + b, \qquad
\mathcal{L}(\beta, b) = \frac1n\sum_i \max\big(0,\ 1 - y_i f(x_i)\big) + \frac{\lambda}{2}\,\beta^\top K\beta .$$

## C1. The kernels and the loss

Implement, with broadcasting and no Python loop over the points: `rbf_kernel(A, B, gamma)` $= \exp(-\gamma\lVert a - b\rVert^2)$
for all pairs (shape `(len(A), len(B))`), `poly_kernel(A, B, degree, coef0)` $= (a^\top b + c_0)^d$, and
`kernel_svm_loss(params, K, y, lam)` with `params = {"beta": ..., "b": ...}` and `K` the Gram matrix of the training set.
*Hint:* $\lVert a - b\rVert^2 = \lVert a\rVert^2 + \lVert b\rVert^2 - 2\,a^\top b$.

In [ ]:
def rbf_kernel(A, B, gamma):
    d2 = jnp.sum(A**2, axis=1)[:, None] + jnp.sum(B**2, axis=1)[None, :] - 2.0 * A @ B.T
    return jnp.exp(-gamma * jnp.maximum(d2, 0.0))


def poly_kernel(A, B, degree, coef0):
    return (A @ B.T + coef0) ** degree


def kernel_svm_loss(params, K, y, lam):
    f = K @ params["beta"] + params["b"]
    return jnp.mean(jnp.maximum(0.0, 1.0 - y * f)) + 0.5 * lam * params["beta"] @ K @ params["beta"]

**How we got there.** Broadcasting turns `(n, 1) + (1, m) - (n, m)` into all pairwise squared distances at once, so no loop
is needed; `jnp.maximum(d2, 0)` removes the tiny negative values of rounding. The polynomial kernel is one matrix product.
For the loss, $f = K\beta + b$ evaluates the model on the training points (the sum over $j$ of the representer form is a
matrix-vector product), and the penalty $\beta^\top K \beta$ is $\lVert f\rVert^2$ in the feature space.

In [ ]:
A_, B_ = jnp.asarray(Xm[:50]), jnp.asarray(Xm[50:90])
compare("rbf kernel (gamma 0.7)", rbf_kernel(A_, B_, 0.7), sk_rbf_kernel(Xm[:50], Xm[50:90], gamma=0.7))
compare("polynomial kernel (d 3, c0 1)", poly_kernel(A_, B_, 3, 1.0), sk_poly_kernel(Xm[:50], Xm[50:90], degree=3, gamma=1.0, coef0=1.0))

ympm = 2.0 * ym - 1
C, gamma = 10.0, 1.0
K_tr = rbf_kernel(jnp.asarray(Xm), jnp.asarray(Xm), gamma)
K_te = rbf_kernel(jnp.asarray(Xm_te), jnp.asarray(Xm), gamma)
p, hist = train(kernel_svm_loss, {"beta": jnp.zeros(len(ym)), "b": jnp.zeros(())}, K_tr, jnp.asarray(ympm), 1 / (C * len(ym)),
                lr=0.02, steps=4000)
f_ours = np.asarray(K_te @ p["beta"] + p["b"])
ref = SVC(kernel="rbf", C=C, gamma=gamma).fit(Xm, ym)
print(f"test accuracy: ours {np.mean((f_ours > 0) == ym_te):.3f}   SVC {ref.score(Xm_te, ym_te):.3f}   "
      f"same prediction on {np.mean((f_ours > 0) == ref.predict(Xm_te)):.1%} of the test points")
print(f"correlation of the decision functions: {np.corrcoef(f_ours, ref.decision_function(Xm_te))[0, 1]:.4f}")

**Answers (C1).** Both kernels agree with scikit-learn to 1e-14 (rounding). The RBF Gram matrix has ones on the diagonal (a point is identical to itself) and values in $(0, 1]$: a value near 0 means two points are far apart relative to $1/\sqrt{\gamma}$ and do not influence each other. The kernel SVM trained by `jax.grad` reaches the test accuracy of `SVC` (0.936 against 0.935), makes the same prediction on 99.8% of the 2000 test points and its decision function has a correlation of 0.997 with SVC's. It is not exactly the same solution (the QP solver stops with exact zeros in $\beta$; gradient descent does not).

**Gamma.** The same loss for four values of $\gamma$ (training accuracy, test accuracy, number of non-negligible $\beta_j$):

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.4))
rows = []
for ax, g in zip(axes, (0.1, 1, 10, 100)):
    Kt = rbf_kernel(jnp.asarray(Xm), jnp.asarray(Xm), g)
    pg, _ = train(kernel_svm_loss, {"beta": jnp.zeros(len(ym)), "b": jnp.zeros(())}, Kt, jnp.asarray(ympm), 1 / (C * len(ym)), lr=0.02, steps=4000)
    def score(P, pg=pg, g=g):
        return np.asarray(rbf_kernel(jnp.asarray(P), jnp.asarray(Xm), g) @ pg["beta"] + pg["b"])

    acc_tr = np.mean((score(Xm) > 0) == ym)
    acc_te = np.mean((score(Xm_te) > 0) == ym_te)
    nz = int(np.sum(np.abs(np.asarray(pg["beta"])) > 0.05 * np.abs(np.asarray(pg["beta"])).max()))
    rows.append((g, float(acc_tr), float(acc_te), nz))
    plot_decision(score, Xm, ym, f"gamma = {g:g}", ax=ax)
plt.tight_layout()
plt.show()
print(pl.DataFrame(rows, schema=["gamma", "train acc", "test acc", "large |beta_j|"], orient="row"))

**Answers (C2).** $\gamma = 0.1$ is *under*fitted (train 0.845, test 0.861: both low, the boundary is almost a line); $\gamma = 1$ is the best (0.950 / 0.936); $\gamma = 100$ is *over*fitted (train 1.000, test 0.884: each point has its own island). The tell-tale sign is the gap between the train and test accuracy together with the island-shaped boundary.

**Choosing $C$ and $\gamma$.** The log-spaced grid with 5-fold cross-validation (scikit-learn's `GridSearchCV`, given) on the
breast-cancer data: with raw features and with the standardized ones.

In [ ]:
Cs, gs = 10.0 ** np.arange(-2, 4), 10.0 ** np.arange(-6, 1)
fig_scores = {}
for label, (A, y_) in {"raw features": (Xb[tr], yb_tr), "standardized": (Xb_tr, yb_tr)}.items():
    t0 = time.time()
    grid = GridSearchCV(SVC(), {"C": Cs, "gamma": gs}, cv=StratifiedKFold(5, shuffle=True, random_state=0), n_jobs=-1).fit(A, y_)
    fig_scores[label] = grid.cv_results_["mean_test_score"].reshape(len(Cs), len(gs))
    print(f"{label:<14}: best {grid.best_params_}, CV accuracy {grid.best_score_:.3f}   ({time.time() - t0:.1f} s)")
for label, sc in fig_scores.items():
    plot_scores_grid(Cs, gs, sc, "gamma (log)", "C (log)", f"CV accuracy, {label}")

**Answers (C3-C4).**

3. With raw features the best cell is $\gamma = 10^{-6}$ (CV accuracy 0.946) and the grid took 3.8 s; with standardized features the best is $C = 10$, $\gamma = 0.01$ (0.979) in 0.3 s. The squared distance is dominated by `mean_area` and `mean_perimeter` (ranges of hundreds and thousands), so only a tiny $\gamma$ keeps the kernel from being 0 everywhere, and the SVM then uses basically two features. Scaling makes the problem both better and cheaper: all features contribute and the optimizer converges quickly.
4. The Gram matrix of $n$ points has $n^2$ entries; for 100 000 training points it needs $10^{10} \times 8$ bytes = **80 GB** in float64, and the solver time grows between $n^2$ and $n^3$. That is why kernel SVMs are used up to some tens of thousands of examples, and linear SVMs or neural networks beyond.

# Part D — Support vector regression

For a numeric target the hinge becomes the **$\varepsilon$-insensitive loss**: errors smaller than $\varepsilon$ cost
nothing, larger ones cost linearly.

## D1. The loss

Implement `eps_insensitive(r, eps)` $= \max(0, |r| - \varepsilon)$ and
`svr_loss(params, X, y, eps, lam)` $=$ mean of `eps_insensitive(y - (X w + b), eps)` $+ \tfrac{\lambda}{2}\lVert w\rVert^2$
with `params = {"w": ..., "b": ...}`.

In [ ]:
def eps_insensitive(r, eps):
    return jnp.maximum(0.0, jnp.abs(r) - eps)


def svr_loss(params, X, y, eps, lam):
    r = y - (X @ params["w"] + params["b"])
    return jnp.mean(eps_insensitive(r, eps)) + 0.5 * lam * jnp.sum(params["w"] ** 2)

**How we got there.** The residual is $r = y - f(x)$; `jnp.abs` makes the loss symmetric and `jnp.maximum(0, ·)` creates the
flat bottom (the tube). Points inside the tube have a zero subgradient: they do not move the model, exactly like the points
outside the margin of the classifier.

A noisy line ($y = 2x + 1$, noise 1) with **three gross outliers on the same side**. Least squares is pulled by them;
the $\varepsilon$-insensitive loss (like the absolute error) is not.

In [ ]:
rng = np.random.default_rng(0)
x1 = np.linspace(0, 10, 40)
y1 = 2 * x1 + 1 + rng.normal(0, 1, 40)
y1[[5, 20, 33]] += [25.0, 30.0, 28.0]
X1 = x1[:, None]
p_svr, _ = train(svr_loss, {"w": jnp.zeros(1), "b": jnp.zeros(())}, X1, y1, 0.5, 1 / (10 * len(y1)), lr=0.1, steps=4000)
ols = LinearRegression().fit(X1, y1)
sk = SVR(kernel="linear", C=10, epsilon=0.5).fit(X1, y1)
print(pl.DataFrame({"model": ["least squares", "ours (jax.grad)", "SVR (scikit-learn)"],
                    "slope": [float(ols.coef_[0]), float(p_svr["w"][0]), float(sk.coef_[0, 0])],
                    "intercept": [float(ols.intercept_), float(p_svr["b"]), float(sk.intercept_[0])]}))
plt.scatter(x1, y1, s=12)
xx = np.linspace(0, 10, 2)
plt.plot(xx, ols.coef_[0] * xx + ols.intercept_, label="least squares")
plt.plot(xx, float(p_svr["w"][0]) * xx + float(p_svr["b"]), label="epsilon-insensitive")
plt.legend()
plt.show()

**Answers (D1).** The true line has slope 2 and intercept 1. Least squares gives 2.053 and **2.748**: the three outliers are all positive (+25, +30, +28) and pull the intercept up by 1.7. The $\varepsilon$-insensitive loss grows linearly, like the absolute error, so each outlier has a bounded influence: slope 2.036 and intercept **0.785** (an error of 0.2). Our `jax.grad` solution and scikit-learn's `SVR` agree to the third decimal. With $\varepsilon = 0$ the loss is the absolute error (a median regression); a larger $\varepsilon$ ignores more small errors and keeps fewer support vectors, which are the points on or outside the tube.

# Part E — A multilayer perceptron in JAX (a practice exercise)

Keras (Part F) is the tool you will use; here you write the smallest network yourself, to see that there is nothing more
to it than *parameters, a forward pass and a loss*. The parameters are a **list of dictionaries**
`[{"W": ..., "b": ...}, ...]`, one per layer: a pytree, so `jax.grad` returns a gradient with the same structure.

## E1. Initialization, forward pass, loss

Implement

* `init_params(key, sizes)`: for `sizes = [64, 32, 10]` two layers with $W \sim \mathcal{N}(0, 2/n_\text{in})$ (He) and
  $b = 0$ (use `jax.random.split` and `jax.random.normal`);
* `forward(params, X)`: ReLU after every layer except the last; returns the **logits**;
* `cross_entropy(params, X, y)`: the mean of $-\log\text{softmax}(\text{logits})_{y}$, computed stably as
  `logsumexp(logits) - logits[y]` (`y` holds the class index).

In [ ]:
def init_params(key, sizes):
    keys = jax.random.split(key, len(sizes) - 1)
    return [{"W": jax.random.normal(k, (m, n)) * jnp.sqrt(2.0 / m), "b": jnp.zeros(n)} for k, m, n in zip(keys, sizes[:-1], sizes[1:])]


def forward(params, X):
    h = X
    for layer in params[:-1]:
        h = jax.nn.relu(h @ layer["W"] + layer["b"])
    return h @ params[-1]["W"] + params[-1]["b"]


def cross_entropy(params, X, y):
    logits = forward(params, X)
    return jnp.mean(jax.nn.logsumexp(logits, axis=1) - jnp.take_along_axis(logits, y[:, None], axis=1)[:, 0])

**How we got there.** *Initialization:* with $n_\text{in}$ inputs of variance 1 the pre-activation has variance
$n_\text{in} \sigma_W^2$; $\sigma_W^2 = 2/n_\text{in}$ keeps it at 2 and the ReLU (which zeroes half) brings it back to the
scale of the input. Zeros would make all units identical. *Forward:* a loop over the layers, the same matrix product as in the
slides. *Loss:* $-\log\text{softmax}_y = \log\sum_c e^{z_c} - z_y$; `logsumexp` subtracts the maximum internally, so a large
logit does not overflow.

**XOR and the digits.** A network without hidden layer cannot solve XOR; one hidden layer can. Then the digits: 64 pixels,
a hidden layer of 32 units, 10 classes, full-batch Adam through `train`.

In [ ]:
rng = np.random.default_rng(0)
Xx = rng.uniform(0.15, 1.0, (400, 2)) * rng.choice([-1.0, 1.0], (400, 2))
yx = (Xx[:, 0] * Xx[:, 1] < 0).astype(int)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, sizes in zip(axes, ([2, 2], [2, 8, 2])):
    p, _ = train(cross_entropy, init_params(jax.random.PRNGKey(0), sizes), jnp.asarray(Xx), jnp.asarray(yx), lr=0.05, steps=1500)
    acc = np.mean(np.asarray(forward(p, jnp.asarray(Xx)).argmax(1)) == yx)
    plot_decision(lambda P, p=p: np.asarray(forward(p, jnp.asarray(P))[:, 1] - forward(p, jnp.asarray(P))[:, 0]), Xx, yx,
                  f"layers {sizes}: accuracy {acc:.2f}", ax=ax)
    print(f"XOR, layers {sizes}: {sum(int(np.prod(q['W'].shape)) + int(np.prod(q['b'].shape)) for q in p)} parameters, accuracy {acc:.3f}")
plt.tight_layout()
plt.show()

params = init_params(jax.random.PRNGKey(1), [64, 32, 10])
print("initial loss:", float(cross_entropy(params, jnp.asarray(Xd_tr), jnp.asarray(yd_tr))), "(random guess: log 10 = 2.303)")
t0 = time.time()
params, hist_jax = train(cross_entropy, params, jnp.asarray(Xd_tr), jnp.asarray(yd_tr), lr=0.01, steps=1000)
acc_jax = float(np.mean(np.asarray(forward(params, jnp.asarray(Xd_te)).argmax(1)) == yd_te))
time_jax = time.time() - t0
print(f"plain JAX MLP [64, 32, 10]: {time_jax:.1f} s, train loss {hist_jax[-1]:.4f}, test accuracy {acc_jax:.3f}")
plot_history({"training loss": hist_jax})

**Answers (E1).**

1. Without a hidden layer the network is a linear classifier: 6 parameters and an accuracy of **0.733**. A line can classify at most three of the four XOR quadrants correctly (0.75). With one hidden layer of 8 ReLU units (42 parameters) the accuracy is **1.000**: the hidden layer builds the features (such as $|x_1 - x_2|$) in which the classes are linearly separable.
2. The initial loss is 2.38, close to $\log 10 = 2.303$: with small random logits the softmax is nearly uniform over 10 classes, so every example costs about $-\log(1/10)$. A loss far above that at the start means the initial weights are too large.
3. The parameter count of `[64, 32, 10]` is $64\cdot 32 + 32 + 32\cdot 10 + 10 = 2410$. After 1000 full-batch Adam steps (3.2 s) the training loss is 0.0028 and the test accuracy **0.960**: the network memorizes the 1347 training digits (train accuracy 1.000).

## E2. Backpropagation by hand

For the network with **one** ReLU hidden layer, `params = [{"W": W1, "b": b1}, {"W": W2, "b": b2}]`, implement
`backward_by_hand(params, X, y)` returning a list with the same structure as `params` (the gradient of `cross_entropy`).
Follow the chain rule from the loss to the input; the forward values you need are $z_1 = XW_1 + b_1$, $h = \text{ReLU}(z_1)$,
the logits $z_2 = hW_2 + b_2$ and $p = \text{softmax}(z_2)$.

*Hints:* $\partial\mathcal{L}/\partial z_2 = (p - \text{onehot}(y))/n$; for $z = hW + b$: $\partial/\partial W = h^\top \delta$,
$\partial/\partial b = \sum_i \delta_i$, $\partial/\partial h = \delta W^\top$; the ReLU passes the gradient where $z_1 > 0$.

In [ ]:
def backward_by_hand(params, X, y):
    W1, b1, W2, b2 = params[0]["W"], params[0]["b"], params[1]["W"], params[1]["b"]
    n = X.shape[0]
    z1 = X @ W1 + b1
    h = jnp.maximum(z1, 0.0)
    p = jax.nn.softmax(h @ W2 + b2, axis=1)
    d2 = (p - jax.nn.one_hot(y, p.shape[1])) / n          # dL/dz2
    dh = d2 @ W2.T                                         # the error goes back through W2
    d1 = dh * (z1 > 0)                                     # ... and through the ReLU
    return [{"W": X.T @ d1, "b": d1.sum(0)}, {"W": h.T @ d2, "b": d2.sum(0)}]

**How we got there.** Each line of the backward pass is one rule: the loss gives $\delta_2 = (p - \text{onehot})/n$ (softmax
and cross-entropy combine into this simple residual); a linear layer sends $\delta$ back with $W^\top$ and accumulates
$h^\top\delta$ for its own weights; the ReLU multiplies by the indicator $z_1 > 0$. Four matrix products, no loop over the
examples.

In [ ]:
small = init_params(jax.random.PRNGKey(2), [64, 16, 10])
Xj, yj = jnp.asarray(Xd_tr[:200]), jnp.asarray(yd_tr[:200])
g_hand = backward_by_hand(small, Xj, yj)
g_auto = jax.grad(cross_entropy)(small, Xj, yj)
for i, (gh, ga) in enumerate(zip(g_hand, g_auto)):
    compare(f"layer {i + 1}: dW", gh["W"], ga["W"])
    compare(f"layer {i + 1}: db", gh["b"], ga["b"])

**Answers (E2).** The hand-written backward pass equals `jax.grad` to 1e-17 for every weight and bias. It costs about two matrix products per layer, once; finite differences would need two forward passes *per parameter* (2 x 2250 here). That is the point of backpropagation: **all** derivatives for the price of a few forward passes.

## E3. Three implementations of the same network

The network of E1 (`[64, 32, 10]`: 64 pixels, 32 ReLU units, 10 classes) written in the three tools you meet in this course:
your JAX code, scikit-learn's `MLPClassifier` and a Keras `Sequential` (JAX backend). Same split, same scaling, same
optimizer (Adam, learning rate 0.01), full batch and 1000 steps. There is no code to write: the driver prints the number of
parameters, the training loss, the test accuracy and how often two models give the same answer.


In [ ]:
if "time_jax" not in globals():
    raise NotImplementedError("finish E1 first: this cell compares your trained network")

Xtr_j, ytr_j = jnp.asarray(Xd_tr), jnp.asarray(yd_tr)
n_tr = len(yd_tr)

t0 = time.time()
sk = MLPClassifier((32,), solver="adam", learning_rate_init=0.01, batch_size=n_tr, max_iter=1000, tol=0.0,
                   n_iter_no_change=1000, alpha=0.0, random_state=0).fit(Xd_tr, yd_tr)
t_sk = time.time() - t0

keras.utils.set_random_seed(0)
kmodel = keras.Sequential([keras.Input((64,)), keras.layers.Dense(32, activation="relu"),
                           keras.layers.Dense(10, activation="softmax")])
kmodel.compile(keras.optimizers.Adam(0.01), "sparse_categorical_crossentropy")
t0 = time.time()
kmodel.fit(Xd_tr, yd_tr, epochs=1000, batch_size=n_tr, verbose=0)
t_keras = time.time() - t0

p_sk = sk.predict_proba(Xd_tr)
loss = {
    "plain JAX (Part E)": float(cross_entropy(params, Xtr_j, ytr_j)),
    "scikit-learn MLPClassifier": float(-np.mean(np.log(p_sk[np.arange(n_tr), yd_tr]))),
    "Keras Sequential (JAX backend)": float(kmodel.evaluate(Xd_tr, yd_tr, verbose=0)),
}
pred = {
    "plain JAX (Part E)": np.asarray(forward(params, jnp.asarray(Xd_te)).argmax(1)),
    "scikit-learn MLPClassifier": sk.predict(Xd_te),
    "Keras Sequential (JAX backend)": kmodel.predict(Xd_te, verbose=0).argmax(1),
}
n_par = {
    "plain JAX (Part E)": sum(int(np.prod(q["W"].shape)) + int(np.prod(q["b"].shape)) for q in params),
    "scikit-learn MLPClassifier": sum(c.size for c in sk.coefs_) + sum(i.size for i in sk.intercepts_),
    "Keras Sequential (JAX backend)": kmodel.count_params(),
}
secs = {"plain JAX (Part E)": float(time_jax), "scikit-learn MLPClassifier": t_sk, "Keras Sequential (JAX backend)": t_keras}
rows = [(k, n_par[k], loss[k], float(np.mean(pred[k] == yd_te)), secs[k]) for k in pred]
print(pl.DataFrame(rows, schema=["implementation", "parameters", "train loss", "test accuracy", "time (s)"], orient="row"))
names = list(pred)
for i in range(3):
    for j in range(i + 1, 3):
        print(f"same prediction, {names[i]} vs {names[j]}: {np.mean(pred[names[i]] == pred[names[j]]):.1%}")

**Answers (E3).**

1. All three have **2410** parameters ($64\cdot32 + 32 + 32\cdot10 + 10$). The test accuracies are 0.960 (plain JAX), 0.960 (`MLPClassifier`) and 0.971 (Keras): at most 0.011 apart, which is 5 of the 450 test images, and the models give the same answer on 98.2-98.7% of them. That is the size of the differences between two seeds of one implementation (Part F, sd about 0.003 with minibatches), so with one run each we cannot rank the tools; repeating with seeds, as in F3, is what would.
2. Three differences: the **initialization** (the plain JAX code uses He normal, Keras `Dense` Glorot uniform, scikit-learn its own Glorot scaling), the **numerical precision** (float32 in JAX and Keras, float64 in scikit-learn) and the **optimizer details** (the Adam epsilon and the order of the operations). On a non-convex loss the initialization decides which minimum is reached, so it is the one we expect to matter most. The time differs by a factor 10 (scikit-learn runs full-batch Adam in NumPy, 22 s; Keras on the JAX backend 2 s): same model, very different cost.

# Part F — Networks with Keras

Keras 3 on the JAX backend is the tool for real work: `Sequential`, `compile` (optimizer and loss), `fit`.

## F1. Build the network

Implement `build_keras_mlp(sizes, n_in, n_out, l2=0.0, dropout=0.0, seed=0)`: a `keras.Sequential` with
`keras.Input((n_in,))`, one `Dense(size, activation="relu", kernel_regularizer=L2(l2) if l2 else None)` per hidden size
(followed by `Dropout(dropout)` if `dropout > 0`), and a final `Dense(n_out, activation="softmax")`.
Call `keras.utils.set_random_seed(seed)` first, so that the weights are reproducible.

In [ ]:
def build_keras_mlp(sizes, n_in, n_out, l2=0.0, dropout=0.0, seed=0):
    keras.utils.set_random_seed(seed)
    reg = keras.regularizers.L2(l2) if l2 else None
    layers = [keras.Input((n_in,))]
    for s in sizes:
        layers.append(keras.layers.Dense(s, activation="relu", kernel_regularizer=reg))
        if dropout > 0:
            layers.append(keras.layers.Dropout(dropout))
    layers.append(keras.layers.Dense(n_out, activation="softmax"))
    return keras.Sequential(layers)

**How we got there.** `Dense(s, activation="relu")` is exactly `relu(h @ W + b)` of Part E; the last layer is the softmax,
so the loss `sparse_categorical_crossentropy` takes probabilities and integer labels. The regularizer adds
$\ell_2\sum W^2$ to the loss (weight decay = a Gaussian prior, Class 04). Dropout is only active in `fit`, never in `predict`.

The driver trains five networks on the digits (training part split again into 80% fit and 20% validation) and compares
them with the plain JAX network of Part E and with scikit-learn's `MLPClassifier`.

In [ ]:
n_fit = int(0.8 * len(yd_tr))
Xf, yf, Xv, yv = Xd_tr[:n_fit], yd_tr[:n_fit], Xd_tr[n_fit:], yd_tr[n_fit:]
configs = {
    "no regularization": dict(),
    "weight decay 1e-3": dict(l2=1e-3),
    "dropout 0.3": dict(dropout=0.3),
    "early stopping": dict(early=True),
    "dropout + early stopping": dict(dropout=0.3, early=True),
}
histories, rows = {}, []
for name, cfg in configs.items():
    early = cfg.pop("early", False)
    net = build_keras_mlp([64, 64], 64, 10, seed=0, **cfg)
    net.compile(keras.optimizers.Adam(1e-3), "sparse_categorical_crossentropy", metrics=["accuracy"])
    cb = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)] if early else []
    t0 = time.time()
    h = net.fit(Xf, yf, epochs=100, batch_size=32, validation_data=(Xv, yv), verbose=0, callbacks=cb)
    histories[name] = np.array(h.history["val_loss"])
    rows.append((name, len(h.history["loss"]), float(h.history["accuracy"][-1]), float(net.evaluate(Xd_te, yd_te, verbose=0)[1]), time.time() - t0))
skl = MLPClassifier((64, 64), max_iter=300, random_state=0).fit(Xf, yf)
rows.append(("scikit-learn MLPClassifier", skl.n_iter_, float(skl.score(Xf, yf)), float(skl.score(Xd_te, yd_te)), np.nan))
rows.append(("plain JAX (Part E, 32 units)", 1000, float(np.mean(np.asarray(forward(params, jnp.asarray(Xd_tr)).argmax(1)) == yd_tr)), acc_jax, np.nan))
print(pl.DataFrame(rows, schema=["network", "epochs", "train acc", "test acc", "time (s)"], orient="row"))
plot_history(histories, ylabel="validation loss", logy=False)

**Answers (F1-F2).**

1. Without regularization the network reaches a training accuracy of 1.000 and a test accuracy of 0.969 after 100 epochs, while its validation loss has been rising since an early epoch (see the plot). Early stopping stops at epoch 54 with the weights of the best validation loss (test 0.973) and saves half of the time.
2. The five Keras networks lie between 0.953 and 0.973 on 450 test images, one image is 0.2%: the differences are of the size of the **seed-to-seed spread** (sd about 0.003, next cell). With these data and one seed, no regularizer is clearly better; the right conclusion is that the problem is easy and that the 0.5-1% differences need several seeds and a paired test to be believed. Dropout 0.3 with early stopping scoring lowest (0.953) is one seed, not a result.

**Seeds.** The same network and data, ten seeds: the test accuracy is a random variable (Class 01).

In [ ]:
accs = []
for seed in range(10):
    net = build_keras_mlp([64, 64], 64, 10, seed=seed)
    net.compile(keras.optimizers.Adam(1e-3), "sparse_categorical_crossentropy")
    net.fit(Xf, yf, epochs=30, batch_size=64, verbose=0)
    accs.append(float(np.mean(net.predict(Xd_te, verbose=0).argmax(1) == yd_te)))
print(f"test accuracy over 10 seeds: mean {np.mean(accs):.4f}, sd {np.std(accs, ddof=1):.4f}, min {np.min(accs):.3f}, max {np.max(accs):.3f}")

**Answers (F3).** Ten seeds give a mean test accuracy of 0.957 with a standard deviation of 0.0026 (range 0.953-0.962). The plain JAX network (0.960, 32 hidden units, full batch, 1000 steps) is inside that range, and scikit-learn's `MLPClassifier` (0.967) as well: the three implementations are the same model; they differ in the initialization, the minibatches, the optimizer settings and the stopping rule, and all of that is within the noise of one training run.

# Part G — Challenge: which learner? (home)

Choose a data set of `datasets/` (`german_credit`, `wine_quality`, `heart_disease`, `spambase`; see
`datasets/README.md`), prepare it with polars (missing values, one-hot encoding, a binary target), and compare **logistic
regression**, the **RBF SVM** and a **Keras MLP** on 10 stratified 75/25 splits with the same standardization, *fitted on the
training part*. Judge the difference with the **corrected resampled $t$-test** (Class 01):
$t = \bar d \big/ \sqrt{(1/J + n_\text{test}/n_\text{train})\,s_d^2}$ with $J-1$ degrees of freedom.

## G1. The test

Implement `corrected_ttest(a, b, ratio)` returning `(t, p)` for the paired scores `a`, `b` of $J$ repetitions
(`ratio = n_test / n_train`; two-sided $p$ from `scipy.stats.t`).

In [ ]:
def corrected_ttest(a, b, ratio):
    d = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    J = len(d)
    var = d.var(ddof=1)
    if var == 0:
        return 0.0, 1.0
    t = d.mean() / np.sqrt((1 / J + ratio) * var)
    return float(t), float(2 * stats.t.sf(abs(t), J - 1))

**How we got there.** The ten test scores come from overlapping training sets, so they are positively correlated and
the naive variance $s_d^2/J$ is too small (too many false "significant" differences). Nadeau and Bengio replace $1/J$ by
$1/J + n_\text{test}/n_\text{train}$. The code is the formula, with the guard for identical scores.

In [ ]:
Xg, yg_raw, _ = __import__("lab05_utils").load_table("wine_quality", "quality")
yg = (yg_raw >= 6).astype(int)
scores = {"logistic": [], "SVM rbf": [], "MLP": []}
for rep in range(10):
    a_idx, b_idx = stratified_split(yg, 0.25, np.random.default_rng(rep))
    m_, s_ = fit_standardizer(Xg[a_idx])
    A, B = standardize(Xg[a_idx], m_, s_), standardize(Xg[b_idx], m_, s_)
    scores["logistic"].append(LogisticRegression(max_iter=2000).fit(A, yg[a_idx]).score(B, yg[b_idx]))
    scores["SVM rbf"].append(SVC().fit(A, yg[a_idx]).score(B, yg[b_idx]))
    net = build_keras_mlp([64, 64], A.shape[1], 2, dropout=0.2, seed=rep)
    net.compile(keras.optimizers.Adam(1e-3), "sparse_categorical_crossentropy")
    net.fit(A, yg[a_idx], epochs=30, batch_size=64, verbose=0)
    scores["MLP"].append(float(np.mean(net.predict(B, verbose=0).argmax(1) == yg[b_idx])))
ratio = len(b_idx) / len(a_idx)
rows = [(k, float(np.mean(v)), float(np.std(v, ddof=1))) for k, v in scores.items()]
print(pl.DataFrame(rows, schema=["model (wine quality >= 6)", "mean accuracy", "sd"], orient="row"))
for a, b in (("SVM rbf", "MLP"), ("SVM rbf", "logistic"), ("MLP", "logistic")):
    t, p = corrected_ttest(scores[a], scores[b], ratio)
    print(f"{a} - {b}: mean difference {np.mean(scores[a]) - np.mean(scores[b]):+.4f}, t = {t:.2f}, p = {p:.3f}")

**Answers (G1).** On the 6497 wines (good = quality $\ge 6$) the mean accuracies over 10 splits are 0.734 (logistic), 0.772 (SVM) and 0.771 (MLP). Both non-linear learners beat the linear one by about 3.8 points with $p < 0.001$ (corrected test, $t = 11.9$ and $7.5$); the SVM and the MLP are the same ($+0.0004$, $p = 0.93$). The ordinary paired $t$-test would give a larger $t$ (it ignores the overlap of the training sets), and so would declare more differences. The conclusion: the data have a non-linear structure that the linear model misses; between the two flexible learners the data cannot choose, so the cost decides (the SVM here trains in 0.7 s, the MLP in 2 s).